In [4]:
from pathlib import Path
# Path provides convenient handling of file and directory paths.
# It is imported here, although this particular evaluation script
# does not directly use Path below.

import torch
# Main PyTorch library.
# Used for loading the trained model, tensors, device management,
# and inference.

from torch.utils.data import DataLoader
# DataLoader loads the evaluation samples in batches.


from dataset import FakeAudioDataset
# Imports your custom dataset.
#
# This dataset:
#
#     audio file
#         ↓
#     extract_fixed_features()
#         ↓
#     [1, 128, 400] spectrogram tensor
#         ↓
#     label
#
# Labels:
#
#     0 = Real
#     1 = Fake


from model import FakeAudioCNN
# Imports the same CNN architecture that was used during training.


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
# Import evaluation metrics from scikit-learn.
#
# These metrics let us evaluate the model beyond simple accuracy.
#
# This is especially important for your project because your
# real/fake dataset is highly imbalanced.


# =========================================================
# CONFIGURATION
# =========================================================

REAL_DIR = "../data/processed/commonvoice"
# Directory containing real Common Voice audio.
#
# IMPORTANT:
# If you want to evaluate using the preprocessed Common Voice
# audio from your earlier pipeline, this should instead be:
#
#     "../data/processed/commonvoice"


FAKE_DIR = "../data/processed/generated"
# Directory containing the XTTS-v2 generated fake audio.


MODEL_PATH = "../models/fake_audio_detector.pt"
# Location of the trained CNN weights.


BATCH_SIZE = 16
# Number of audio samples evaluated at one time.
#
# Example:
#
#     16 audio files
#         ↓
#     CNN
#         ↓
#     16 predictions


# =========================================================
# DEVICE
# =========================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
# Select the device used for inference.
#
# If CUDA is available:
#
#     GPU
#
# Otherwise:
#
#     CPU
#
# On your Mac, this will likely currently use CPU.


print("Using device:", device)


# =========================================================
# LOAD DATASET
# =========================================================

dataset = FakeAudioDataset(
    real_dir=REAL_DIR,
    fake_dir=FAKE_DIR
)
# Create the complete evaluation dataset.
#
# The dataset contains:
#
#     Real audio → label 0
#     Fake audio → label 1
#
# Each audio file is converted into a fixed-size
# spectrogram when __getitem__() is called.


print("Total evaluation samples:", len(dataset))
# Display the total number of samples being evaluated.


# =========================================================
# DATA LOADER
# =========================================================

data_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)
# Create batches for evaluation.
#
# shuffle=False is intentional.
#
# During evaluation, we do not need to randomize the samples.
# We simply want to evaluate every sample.


# =========================================================
# LOAD MODEL
# =========================================================

model = FakeAudioCNN()
# Create the same CNN architecture used during training.
#
# IMPORTANT:
# The architecture must match the architecture that produced
# fake_audio_detector.pt.


model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)
# Load the learned weights from the saved .pt file.
#
# map_location=device ensures that the model can be loaded
# correctly even if it was trained on a different device.
#
# For example:
#
#     trained on GPU
#          ↓
#     evaluated on CPU
#
# map_location handles that device difference.


model = model.to(device)
# Move the CNN to the selected device.


model.eval()




print("Model loaded successfully.")


# =========================================================
# EVALUATION
# =========================================================

all_labels = []
# Will contain the TRUE labels for every evaluation sample.
#
# Example:
#
#     [0, 0, 1, 0, 1, ...]


all_predictions = []
# Will contain the CNN's PREDICTED labels.
#
# Example:
#
#     [0, 0, 1, 1, 1, ...]


with torch.no_grad():
    # Disable gradient calculation.
    #
    # We are only evaluating the model.
    # We are NOT updating its weights.
    #
    # This reduces memory usage and makes inference faster.


    for features, labels in data_loader:
        # Process the evaluation data batch by batch.
        #
        # features:
        #
        #     [batch_size, 1, 128, 400]
        #
        # labels:
        #
        #     [batch_size]


        features = features.to(device)
        # Move the spectrogram features to CPU/GPU.


        outputs = model(features)
        # Run the batch through the CNN.
        #
        # Output shape:
        #
        #     [batch_size, 2]
        #
        # The two output values represent the model's scores
        # for the two classes:
        #
        #     0 = Real
        #     1 = Fake


        predictions = torch.argmax(
            outputs,
            dim=1
        )
        # Select the class with the highest score.
        #
        # Example:
        #
        #     [2.5, 0.8] → 0 → Real
        #
        #     [0.3, 4.1] → 1 → Fake


        all_labels.extend(
            labels.numpy()
        )
        # Add the true labels from this batch to the complete
        # list of true labels.


        all_predictions.extend(
            predictions.cpu().numpy()
        )
        # Move predictions back to CPU and convert them
        # from PyTorch tensors into NumPy arrays.
        #
        # The result is added to all_predictions.


# =========================================================
# METRICS
# =========================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)
# Accuracy measures the percentage of ALL predictions
# that were correct.
#
# Formula:
#
#     correct predictions
#     -------------------
#     total predictions
#
# However, accuracy can be misleading with imbalanced data.


precision = precision_score(
    all_labels,
    all_predictions,
    zero_division=0
)
# Precision answers:
#
#     "When the model says FAKE,
#      how often is it actually fake?"
#
# Formula:
#
#             True Positives
#     ----------------------------
#     True Positives + False Positives
#
# High precision means the model does not frequently
# falsely accuse real audio of being fake.
#
# zero_division=0 prevents an error if the model predicts
# no fake samples at all.


recall = recall_score(
    all_labels,
    all_predictions,
    zero_division=0
)
# Recall answers:
#
#     "Of all the actual FAKE audio,
#      how much did the model detect?"
#
# Formula:
#
#          True Positives
#     ------------------------
#     True Positives + False Negatives
#
# High recall is especially important for a fake-audio
# detection system because missing synthetic audio is
# a major failure.


f1 = f1_score(
    all_labels,
    all_predictions,
    zero_division=0
)
# F1 score combines precision and recall.
#
# It is the harmonic mean of precision and recall.
#
# F1 is particularly useful when the classes are imbalanced.


# =========================================================
# RESULTS
# =========================================================

print()
print("=" * 60)
print("FAKE AUDIO DETECTION RESULTS")
print("=" * 60)
# Print a readable results header.


print(
    f"Accuracy : {accuracy:.4f}"
)
# Display overall accuracy.
#
# Example:
#
#     Accuracy : 0.9235
#
# means 92.35% of all predictions were correct.


print(
    f"Precision: {precision:.4f}"
)
# Display fake-class precision.


print(
    f"Recall   : {recall:.4f}"
)
# Display fake-class recall.


print(
    f"F1 Score : {f1:.4f}"
)
# Display the F1 score for the fake class.


# =========================================================
# CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)
# Create a 2 × 2 confusion matrix.
#
# With:
#
#     0 = Real
#     1 = Fake
#
# the matrix is:
#
#                 Predicted
#                 Real   Fake
#
# Actual Real      TN     FP
# Actual Fake      FN     TP
#
#
# TN = True Negative
#     Real correctly classified as Real
#
# FP = False Positive
#     Real incorrectly classified as Fake
#
# FN = False Negative
#     Fake incorrectly classified as Real
#
# TP = True Positive
#     Fake correctly classified as Fake


print()
print("Confusion Matrix:")
print(cm)
# Display the confusion matrix.


# =========================================================
# CLASSIFICATION REPORT
# =========================================================

print()
print("Classification Report:")
# Print a readable header.


print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Real",
            "Fake"
        ],
        zero_division=0
    )
)
# Generate a detailed report for BOTH classes.
#
# It reports:
#
#     precision
#     recall
#     F1-score
#     support
#
# "support" = number of actual samples belonging to each class.
#
# Because we specify:
#
#     target_names=["Real", "Fake"]
#
# the report will explicitly identify the two classes.



Using device: cpu

Real audio directory:
/Users/melaniequ/Voice Cloning and Fake Audio Detection/data/processed/commonvoice

Fake audio directory:
/Users/melaniequ/Voice Cloning and Fake Audio Detection/data/processed/generated

Original real files: 7432
Original fake files: 741
Skipping short audio (0.160s): ../data/processed/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-aln/audios/spontaneous-speech-aln-46236.mp3
Skipping short audio (0.160s): ../data/processed/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-aln/audios/spontaneous-speech-aln-46208.mp3
Skipping short audio (0.160s): ../data/processed/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-aln/audios/spontaneous-speech-aln-46235.mp3
Skipping short audio (0.156s): ../data/processed/commonvoice/train/mcv-sps-st-09-2025/sps-corpus-1.0-2025-09-05-aln/audios/spontaneous-speech-aln-46055.mp3
Skipping short audio (0.160s): ../data/processed/commonvoice/train/mcv-sps-st-09-2025/sps-corpus

/var/folders/0g/jtvhdh2x0xl58z5r29424k1m0000gn/T/ipykernel_86581/88439842.py:166: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(



FAKE AUDIO DETECTION RESULTS
Accuracy : 0.9998
Precision: 0.9973
Recall   : 1.0000
F1 Score : 0.9987

Confusion Matrix:
[[7416    2]
 [   0  741]]

Classification Report:
              precision    recall  f1-score   support

        Real       1.00      1.00      1.00      7418
        Fake       1.00      1.00      1.00       741

    accuracy                           1.00      8159
   macro avg       1.00      1.00      1.00      8159
weighted avg       1.00      1.00      1.00      8159

